In [9]:
from sound_simulation import *
from beamforming import *
from sensor_array import *
from scene import *
from util import *

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Audio

In [16]:
duration = 5
sr = 48000
c = 344
fmin = 500
fmax = 6000

sensor_array = array_circle8()
noise = prepare_audio("../samples/meas1_ambience.wav", duration, sr, False)

target_position = np.array([50, 0, 10])
scene = scene_stationary(duration, sr, target_position)
target_sound = prepare_drone_audio(scene, sensor_array, "../samples/mavic4_flight.wav", duration, sr, db2lin(65-60))

audio = noise + target_sound

In [32]:

N = 2**10
window = np.hanning(N)**0.5
VN, f, validf = valid_frequency_bins(fmin, fmax, N, sr)
SN, tidx, snaps = stft(audio, window, N//2, validf)

k = 2 * np.pi * f / c

target_udir = target_position / np.linalg.norm(target_position)
target_steer = create_steering_vectors(create_distance_vectors(target_udir[None,:], sensor_array.positions), k[:,None])

w_das = das_weights(target_steer)
filtered_snaps = np.array([beamform_wideband(s.T, w_das) for s in snaps])

sdas = istft(filtered_snaps, window, tidx, validf)
ch0 = istft(snaps[:,0,:], window, tidx, validf)

display(Audio(sdas, rate=sr))
display(Audio(ch0, rate=sr))

# What next?